# Cahn-Hilliard PINN — PyTorch Implementation

**PDE (Equation 6, PhysRevE.106.065303):**

$$\frac{\partial U}{\partial t} = D\,\nabla^2\!\bigl(\nabla^2 U + a_2\,U + a_4\,U^3\bigr)$$

**Approach:** Physics-Informed Neural Network trained in two phases:
1. **Adam** warmup (first-order, fast)
2. **Self-Scaled Broyden** quasi-Newton (second-order, precise)

All hyperparameters, PDE coefficients, domain bounds, and training options
are loaded from a **YAML configuration file** so that experiments can be
launched by simply swapping configs — no code edits required.

In [ ]:
# ============================================================================
# CELL 1: IMPORTS AND ENVIRONMENT SETUP
# ============================================================================

import os
import math
import yaml
from time import perf_counter

import numpy as np
import scipy.io
from scipy.optimize import minimize
from scipy.linalg import cholesky, LinAlgError
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import gaussian_filter

import torch
import torch.nn as nn
from torch.nn.utils import parameters_to_vector, vector_to_parameters

import matplotlib.pyplot as plt
from matplotlib import cm

# ---- Path to the YAML configuration file -----------------------------------
# Change ONLY this line to switch experiments.
CONFIG_PATH = "configs/cahn_hilliard_canonical.yaml"

In [ ]:
# ============================================================================
# CELL 2: LOAD CONFIGURATION AND SET ALL PARAMETERS
# ============================================================================
# Every tunable quantity is read from the YAML config.  Nothing is hardcoded.
# ============================================================================

with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

# --- Equation coefficients ---
D  = cfg["equation"]["D"]      # Diffusion / mobility constant
a2 = cfg["equation"]["a2"]     # Quadratic coeff in chemical potential
a4 = cfg["equation"]["a4"]     # Quartic coeff in chemical potential

# --- Spatial / temporal domain ---
x_min = cfg["domain"]["x_min"]
x_max = cfg["domain"]["x_max"]
y_min = cfg["domain"]["y_min"]
y_max = cfg["domain"]["y_max"]
t_min = cfg["domain"]["t_min"]
t_max = cfg["domain"]["t_max"]
Lx = x_max - x_min             # Domain length in x
Ly = y_max - y_min             # Domain length in y
normalize_inputs = cfg["domain"].get("normalize_inputs", True)

# --- Initial condition ---
ic_cfg        = cfg["initial_condition"]
ic_type       = ic_cfg["type"]
ic_seed       = ic_cfg.get("seed", 42)
ic_grid_nx    = ic_cfg.get("grid_nx", 128)
ic_grid_ny    = ic_cfg.get("grid_ny", 128)
ic_smooth_sig = ic_cfg.get("smoothing_sigma", 2.0)

# --- Boundary conditions ---
bc_cfg         = cfg["boundary_conditions"]
bc_type        = bc_cfg["type"]               # "periodic"
bc_enforcement = bc_cfg.get("enforcement", "soft")
bc_deriv_order = bc_cfg.get("derivative_order", 0)

# --- Network ---
net_cfg        = cfg["network"]
layer_dims     = tuple(net_cfg["layer_dims"])
activation_name = net_cfg.get("activation", "tanh")
output_scaling = net_cfg.get("output_scaling", 1.0)

# --- Training schedule ---
adam_cfg = cfg["training"]["adam"]
bfgs_cfg = cfg["training"]["bfgs"]

Nepochs_ADAM   = adam_cfg["epochs"]
adam_lr        = adam_cfg["lr"]
adam_betas     = tuple(adam_cfg["betas"])
adam_eps        = adam_cfg["eps"]
lr_decay_rate  = adam_cfg.get("lr_decay_rate", 0.98)
lr_decay_steps = adam_cfg.get("lr_decay_steps", 1000)

Nbfgs          = bfgs_cfg["epochs"]
Nchange        = bfgs_cfg["batch_size"]
bfgs_method    = bfgs_cfg["method"]               # "BFGS" or "L-BFGS-B"
bfgs_variant   = bfgs_cfg.get("variant", "none")  # SSBroyden2, etc. (BFGS only)
bfgs_init_scale = bfgs_cfg.get("initial_scale", False)
bfgs_power     = bfgs_cfg.get("power", 1.0)
bfgs_maxcor    = bfgs_cfg.get("maxcor", 20)       # L-BFGS-B: num corrections

# --- Sampling ---
samp_cfg   = cfg["sampling"]
Nint       = samp_cfg["n_interior"]
N0         = samp_cfg["n_initial"]
Nb         = samp_cfg["n_boundary"]
Nresample  = samp_cfg["resample_every"]
rad_cfg    = samp_cfg.get("rad", {})
rad_on     = rad_cfg.get("enabled", True)
k1         = rad_cfg.get("k1", 1.0)
k2         = rad_cfg.get("k2", 1.0)
Nsampling  = rad_cfg.get("n_candidates", 50000)
rad_args   = (k1, k2)

# --- Loss weights ---
lw         = cfg["loss_weights"]
lam_pde    = lw.get("pde", 1.0)
lam_ic     = lw.get("ic", 5.0)
lam_bc     = lw.get("bc", 5.0)

# --- Logging ---
log_cfg    = cfg.get("logging", {})
Nprint     = log_cfg.get("print_every", 100)
RESULTS_DIR = log_cfg.get("results_dir", "results_cahn_hilliard")

# --- Global ---
SEED       = cfg.get("seed", 2)
dtype_str  = cfg.get("dtype", "float64")

# ---------- Apply settings ----------
torch_dtype = torch.float64 if dtype_str == "float64" else torch.float32
torch.set_default_dtype(torch_dtype)
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Config : {cfg['experiment']['name']}")
print(f"  PDE  : dU/dt = {D}*lap(lap U + {a2}*U + {a4}*U^3)")
print(f"  Domain: x=[{x_min},{x_max}], y=[{y_min},{y_max}], t=[{t_min},{t_max}]")
print(f"  Net  : {layer_dims}  activation={activation_name}")
if Nbfgs > 0:
    print(f"  Train: Adam({Nepochs_ADAM}) -> {bfgs_variant if bfgs_method == 'BFGS' else bfgs_method}({Nbfgs})")
else:
    print(f"  Train: Adam-only({Nepochs_ADAM})")
print(f"  Samp : Nint={Nint}, N0={N0}, Nb={Nb}, RAD={'on' if rad_on else 'off'}")

In [ ]:
# ============================================================================
# CELL 3: NEURAL NETWORK DEFINITION
# ============================================================================
# Fully configurable MLP:
#   - Optional input normalisation (maps physical coords → [-1, 1])
#   - Arbitrary depth / width from config
#   - Configurable activation function
#   - Optional output scaling
#   - Custom variance-scaling init on the final linear layer
# ============================================================================

# ---- Activation function lookup -------------------------------------------
ACTIVATIONS = {
    "tanh": nn.Tanh, "relu": nn.ReLU, "gelu": nn.GELU,
    "silu": nn.SiLU, "sigmoid": nn.Sigmoid,
}


class InputNormalization(nn.Module):
    """Non-trainable affine layer: maps [lo, hi] → [-1, 1] per feature."""
    def __init__(self, lo, hi):
        super().__init__()
        lo = torch.tensor(lo, dtype=torch.get_default_dtype())
        hi = torch.tensor(hi, dtype=torch.get_default_dtype())
        self.register_buffer("shift", 0.5 * (lo + hi))     # midpoint
        self.register_buffer("scale", 2.0 / (hi - lo))     # 1 / half-range
    def forward(self, x):
        return (x - self.shift) * self.scale


class OutputScaling(nn.Module):
    """Multiply network output by a fixed constant."""
    def __init__(self, s):
        super().__init__()
        self.s = s
    def forward(self, x):
        return x * self.s


def variance_scaling_init(linear, scale=1.0):
    """TF-style VarianceScaling(mode='fan_avg', distribution='uniform')."""
    fi, fo = linear.in_features, linear.out_features
    limit = math.sqrt(3.0 * scale / (0.5 * (fi + fo)))
    with torch.no_grad():
        linear.weight.uniform_(-limit, limit)
        if linear.bias is not None:
            linear.bias.zero_()


class CahnHilliardNet(nn.Module):
    """
    Configurable MLP for the Cahn-Hilliard PINN.

    Input  : (t, x, y) — 3 features
    Output : U(t, x, y) — 1 feature

    Architecture (canonical):
      InputNorm → Linear(3→40) → tanh → [Linear(40→40) → tanh] × 3
                → Linear(40→1) → OutputScaling
    """
    def __init__(self, layer_dims, activation_name="tanh",
                 output_scale=1.0, normalize=False, input_bounds=None):
        super().__init__()
        layers = []

        # Optional input normalization
        if normalize and input_bounds is not None:
            lo, hi = input_bounds
            layers.append(InputNormalization(lo, hi))

        # Hidden + output layers
        act_cls = ACTIVATIONS.get(activation_name, nn.Tanh)
        for i in range(len(layer_dims) - 1):
            layers.append(nn.Linear(layer_dims[i], layer_dims[i + 1]))
            if i < len(layer_dims) - 2:          # activation after every hidden layer
                layers.append(act_cls())

        # Output scaling
        if output_scale != 1.0:
            layers.append(OutputScaling(output_scale))

        self.net = nn.Sequential(*layers)

        # Custom init for the LAST linear layer
        for m in reversed(list(self.net.modules())):
            if isinstance(m, nn.Linear):
                sc = 1.0 / (output_scale ** 2) if output_scale != 0 else 1.0
                variance_scaling_init(m, scale=sc)
                break

    def forward(self, x):
        return self.net(x)


# ---- Build the network ----------------------------------------------------
input_bounds = ([t_min, x_min, y_min], [t_max, x_max, y_max])
net = CahnHilliardNet(
    layer_dims,
    activation_name=activation_name,
    output_scale=output_scaling,
    normalize=normalize_inputs,
    input_bounds=input_bounds,
).to(DEVICE)

n_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
hess_gb = n_params ** 2 * 8 / 1e9
print(f"Trainable parameters : {n_params}")
print(f"Hessian approx memory: {n_params}x{n_params} = {n_params**2:,} entries  ({hess_gb:.2f} GB)")

In [ ]:
# ============================================================================
# CELL 4: INITIAL CONDITION AND SAMPLING FUNCTIONS
# ============================================================================
# - IC: smoothed random field on a grid, interpolated for arbitrary (x, y)
# - Interior sampling: uniform random (t, x, y) in the domain
# - IC sampling: (t_min, x, y) for enforcing U(0, x, y) = U₀
# - BC sampling: paired points on opposite faces for periodic matching
# - RAD: Residual-Adaptive Distribution for smart resampling
# ============================================================================

# ---- Build the initial condition interpolator ------------------------------
def build_ic_interpolator(ic_cfg, x_min, x_max, y_min, y_max):
    """
    Generate a random IC field on a grid and return a smooth interpolator.

    The field is optionally Gaussian-smoothed so that the PINN (a smooth
    neural network) can represent it.  The interpolator uses periodic
    boundary padding so queries near the domain edges wrap correctly.
    """
    rng   = np.random.RandomState(ic_cfg.get("seed", 42))
    nx    = ic_cfg.get("grid_nx", 128)
    ny    = ic_cfg.get("grid_ny", 128)
    sigma = ic_cfg.get("smoothing_sigma", 2.0)
    low   = ic_cfg.get("low", -1.0)
    high  = ic_cfg.get("high", 1.0)

    # Raw random field on the grid
    field = rng.uniform(low, high, size=(ny, nx))

    # Smooth for PINN compatibility (mode='wrap' respects periodic BCs)
    if sigma > 0:
        field = gaussian_filter(field, sigma=sigma, mode="wrap")

    # Grid coordinates (cell centres)
    dx = (x_max - x_min) / nx
    dy = (y_max - y_min) / ny
    xs = np.linspace(x_min + 0.5 * dx, x_max - 0.5 * dx, nx)
    ys = np.linspace(y_min + 0.5 * dy, y_max - 0.5 * dy, ny)

    # Pad for periodic interpolation near edges
    xs_ext = np.concatenate([[x_min], xs, [x_max]])
    ys_ext = np.concatenate([[y_min], ys, [y_max]])
    field_ext = np.pad(field, ((1, 1), (1, 1)), mode="wrap")

    interp = RegularGridInterpolator(
        (ys_ext, xs_ext), field_ext,
        method="cubic", bounds_error=False, fill_value=None,
    )
    return interp, field


ic_interp, ic_field = build_ic_interpolator(ic_cfg, x_min, x_max, y_min, y_max)


def eval_ic(X_t0):
    """Evaluate the initial condition at (x, y) points (t column ignored).

    Args:
        X_t0: tensor of shape (N, 3) with columns (t, x, y).
    Returns:
        tensor of shape (N, 1) — the IC values U₀(x, y).
    """
    x_np = X_t0[:, 1].detach().cpu().numpy()
    y_np = X_t0[:, 2].detach().cpu().numpy()
    # RegularGridInterpolator was built with axes (y, x)
    vals = ic_interp(np.column_stack([y_np, x_np]))
    return torch.tensor(vals, dtype=torch.get_default_dtype()).reshape(-1, 1)


# ---- Sampling functions ----------------------------------------------------

def sample_interior(n):
    """Random (t, x, y) inside the domain for PDE residual."""
    t = t_min + (t_max - t_min) * np.random.rand(n)
    x = x_min + Lx * np.random.rand(n)
    y = y_min + Ly * np.random.rand(n)
    return torch.tensor(np.column_stack([t, x, y]),
                        dtype=torch.get_default_dtype())


def sample_initial(n):
    """Random (t_min, x, y) for initial-condition enforcement."""
    t = np.full(n, t_min)
    x = x_min + Lx * np.random.rand(n)
    y = y_min + Ly * np.random.rand(n)
    return torch.tensor(np.column_stack([t, x, y]),
                        dtype=torch.get_default_dtype())


def sample_boundary_periodic(n):
    """
    Sample paired boundary points for periodic-BC enforcement.

    Returns 4 tensors of shape (n // 2, 3):
        X_xlo, X_xhi  — pairs along x-boundaries (x_min ↔ x_max)
        X_ylo, X_yhi  — pairs along y-boundaries (y_min ↔ y_max)

    At each pair the network output (and optionally its normal derivative)
    must match → enforced through the loss function.
    """
    n2 = max(n // 2, 1)

    # x-periodic pairs
    t_x = t_min + (t_max - t_min) * np.random.rand(n2)
    y_x = y_min + Ly * np.random.rand(n2)
    X_xlo = torch.tensor(np.column_stack([t_x, np.full(n2, x_min), y_x]),
                          dtype=torch.get_default_dtype())
    X_xhi = torch.tensor(np.column_stack([t_x, np.full(n2, x_max), y_x]),
                          dtype=torch.get_default_dtype())

    # y-periodic pairs
    t_y = t_min + (t_max - t_min) * np.random.rand(n2)
    x_y = x_min + Lx * np.random.rand(n2)
    X_ylo = torch.tensor(np.column_stack([t_y, x_y, np.full(n2, y_min)]),
                          dtype=torch.get_default_dtype())
    X_yhi = torch.tensor(np.column_stack([t_y, x_y, np.full(n2, y_max)]),
                          dtype=torch.get_default_dtype())

    return X_xlo, X_xhi, X_ylo, X_yhi


print(f"IC field range: [{ic_field.min():.3f}, {ic_field.max():.3f}]  "
      f"shape: {ic_field.shape}  smoothing sigma: {ic_smooth_sig}")

In [ ]:
# ============================================================================
# CELL 5: PDE RESIDUAL — 4TH-ORDER AUTOGRAD COMPUTATION
# ============================================================================
#
# PDE:  ∂U/∂t = D · ∇²( ∇²U + a₂·U + a₄·U³ )
#
# Expanding in 2-D:
#
#   ∂U/∂t = D · [ ∇⁴U  +  a₂·∇²U  +  a₄·∇²(U³) ]
#
# where
#   ∇⁴U         = U_xxxx + 2·U_xxyy + U_yyyy            (biharmonic)
#   ∇²U         = U_xx + U_yy                            (Laplacian)
#   ∇²(U³)      = 6U(U_x² + U_y²) + 3U²(U_xx + U_yy)   (analytical)
#
# Using the analytical expansion of ∇²(U³) avoids 3 extra autograd calls.
#
# Derivative chain (8 autograd.grad calls total):
#   1. grad(U, X)     → U_t, U_x, U_y
#   2. grad(U_x, X)   → U_xx  (+ U_xy)
#   3. grad(U_y, X)   → U_yy
#   4. grad(U_xx, X)  → U_xxx, U_xxy
#   5. grad(U_yy, X)  → U_yyy
#   6. grad(U_xxx, X) → U_xxxx
#   7. grad(U_xxy, X) → U_xxyy
#   8. grad(U_yyy, X) → U_yyyy
# ============================================================================


def forward_pass(net, X):
    """Network forward pass. Returns U of shape (N, 1)."""
    return net(X)[:, 0:1]


def compute_pde_residual(net, X):
    """
    Compute the Cahn-Hilliard PDE residual at interior collocation points.

    Args:
        net : the neural network
        X   : tensor (N, 3) of (t, x, y) points with requires_grad enabled

    Returns:
        U        : network prediction, shape (N, 1)
        residual : PDE residual,       shape (N,)
    """
    X = X.clone().detach().requires_grad_(True)
    U = forward_pass(net, X)                  # (N, 1)
    ones = torch.ones_like(U)
    ones1 = torch.ones_like(U[:, 0])          # flat version for 1-D grads

    # ---- 1st derivatives: U_t, U_x, U_y -----------------------------------
    dU  = torch.autograd.grad(U, X, grad_outputs=ones,
                              create_graph=True, retain_graph=True)[0]
    U_t = dU[:, 0]
    U_x = dU[:, 1]
    U_y = dU[:, 2]

    # ---- 2nd derivatives: U_xx, U_yy ---------------------------------------
    dUx = torch.autograd.grad(U_x, X, grad_outputs=ones1,
                              create_graph=True, retain_graph=True)[0]
    U_xx = dUx[:, 1]                         # ∂²U/∂x²

    dUy = torch.autograd.grad(U_y, X, grad_outputs=ones1,
                              create_graph=True, retain_graph=True)[0]
    U_yy = dUy[:, 2]                         # ∂²U/∂y²

    # ---- 3rd derivatives: U_xxx, U_xxy, U_yyy ------------------------------
    dUxx = torch.autograd.grad(U_xx, X, grad_outputs=ones1,
                               create_graph=True, retain_graph=True)[0]
    U_xxx = dUxx[:, 1]                        # ∂³U/∂x³
    U_xxy = dUxx[:, 2]                        # ∂³U/∂x²∂y

    dUyy = torch.autograd.grad(U_yy, X, grad_outputs=ones1,
                               create_graph=True, retain_graph=True)[0]
    U_yyy = dUyy[:, 2]                        # ∂³U/∂y³

    # ---- 4th derivatives: U_xxxx, U_xxyy, U_yyyy --------------------------
    dUxxx = torch.autograd.grad(U_xxx, X, grad_outputs=ones1,
                                create_graph=True, retain_graph=True)[0]
    U_xxxx = dUxxx[:, 1]                      # ∂⁴U/∂x⁴

    dUxxy = torch.autograd.grad(U_xxy, X, grad_outputs=ones1,
                                create_graph=True, retain_graph=True)[0]
    U_xxyy = dUxxy[:, 2]                     # ∂⁴U/∂x²∂y²

    dUyyy = torch.autograd.grad(U_yyy, X, grad_outputs=ones1,
                                create_graph=True, retain_graph=True)[0]
    U_yyyy = dUyyy[:, 2]                      # ∂⁴U/∂y⁴

    # ---- Assemble PDE terms ------------------------------------------------
    biharmonic = U_xxxx + 2.0 * U_xxyy + U_yyyy          # ∇⁴U
    laplacian  = U_xx + U_yy                               # ∇²U

    # ∇²(U³) via chain rule (avoids 3 extra autograd calls):
    #   ∂²(U³)/∂x² = 6U·(U_x)² + 3U²·U_xx
    #   ∂²(U³)/∂y² = 6U·(U_y)² + 3U²·U_yy
    Uf = U[:, 0]                                           # flatten (N,1)→(N,)
    lap_U3 = 6.0 * Uf * (U_x ** 2 + U_y ** 2) + 3.0 * Uf ** 2 * laplacian

    # Full RHS:  D · [ ∇⁴U + a₂·∇²U + a₄·∇²(U³) ]
    rhs = D * (biharmonic + a2 * laplacian + a4 * lap_U3)

    # Residual: should be ≈ 0 if the network satisfies the PDE
    residual = U_t - rhs

    return U, residual


def compute_bc_derivatives(net, X):
    """Compute U and ∂U/∂n (normal derivative) at boundary points.

    Used when bc_deriv_order >= 1 to enforce derivative periodicity.
    Returns U (N,1), U_x (N,), U_y (N,).
    """
    X = X.clone().detach().requires_grad_(True)
    U = forward_pass(net, X)
    ones = torch.ones_like(U)
    dU = torch.autograd.grad(U, X, grad_outputs=ones,
                             create_graph=True, retain_graph=True)[0]
    return U, dU[:, 1], dU[:, 2]

In [ ]:
# ============================================================================
# CELL 6: LOSS FUNCTION, GRADIENT COMPUTATION, AND TRAINING UTILITIES
# ============================================================================
# Loss = λ_pde · L_PDE  +  λ_ic · L_IC  +  λ_bc · L_BC
#
# L_PDE : MSE of PDE residual at interior points
# L_IC  : MSE of (network − true IC) at t = 0
# L_BC  : MSE of (U_left − U_right) + (U_bottom − U_top)  for periodicity
#         Optionally + derivative matching if bc_deriv_order >= 1
# ============================================================================

mse = nn.MSELoss()


def compute_loss(net, X_int, X_ic, bc_data):
    """
    Full PINN loss for the Cahn-Hilliard equation.

    Args:
        net   : the neural network
        X_int : (N_int, 3) interior collocation points
        X_ic  : (N_ic, 3) initial-condition points (t = t_min)
        bc_data : tuple (X_xlo, X_xhi, X_ylo, X_yhi) boundary pairs

    Returns:
        total_loss : scalar tensor
        pde_loss   : scalar (for logging)
    """
    X_xlo, X_xhi, X_ylo, X_yhi = bc_data

    # ---- PDE residual loss -------------------------------------------------
    _, residual = compute_pde_residual(net, X_int)
    zeros_r = torch.zeros_like(residual)
    L_pde = mse(residual, zeros_r)

    # ---- Initial-condition loss --------------------------------------------
    U_ic_pred = forward_pass(net, X_ic)
    U_ic_true = eval_ic(X_ic).to(U_ic_pred.device)
    L_ic = mse(U_ic_pred, U_ic_true)

    # ---- Periodic boundary-condition loss ----------------------------------
    if bc_deriv_order >= 1:
        # Value + first-derivative matching
        U_lo_x, Ux_lo, _ = compute_bc_derivatives(net, X_xlo)
        U_hi_x, Ux_hi, _ = compute_bc_derivatives(net, X_xhi)
        U_lo_y, _, Uy_lo  = compute_bc_derivatives(net, X_ylo)
        U_hi_y, _, Uy_hi  = compute_bc_derivatives(net, X_yhi)
        L_bc = (mse(U_lo_x, U_hi_x) + mse(U_lo_y, U_hi_y)
                + mse(Ux_lo, Ux_hi) + mse(Uy_lo, Uy_hi))
    else:
        # Value matching only
        U_lo_x = forward_pass(net, X_xlo)
        U_hi_x = forward_pass(net, X_xhi)
        U_lo_y = forward_pass(net, X_ylo)
        U_hi_y = forward_pass(net, X_yhi)
        L_bc = mse(U_lo_x, U_hi_x) + mse(U_lo_y, U_hi_y)

    # ---- Weighted total ----------------------------------------------------
    total = lam_pde * L_pde + lam_ic * L_ic + lam_bc * L_bc
    return total, float(L_pde.detach())


# ---- Adam training step ----------------------------------------------------

def adam_step(net, optimizer, X_int, X_ic, bc_data):
    """One Adam training step: forward → loss → backward → update."""
    for p in net.parameters():
        p.grad = None
    loss_val, pde_val = compute_loss(net, X_int, X_ic, bc_data)
    loss_val.backward()
    optimizer.step()
    return float(loss_val.detach()), pde_val


# ---- SciPy-compatible loss + gradient wrapper (for BFGS) -------------------

def scipy_loss_and_grad(weights, net, X_int, X_ic, bc_data, power=1.0):
    """
    Flat numpy interface for scipy.optimize.minimize.

    1. Load weights into network
    2. Compute loss + gradients via autograd
    3. Return (scalar_loss, flat_gradient) as numpy arrays

    The `power` arg: minimise L^(1/power) instead of L (conditioning trick).
    """
    device = next(net.parameters()).device
    dtype  = next(net.parameters()).dtype
    w = torch.as_tensor(weights, dtype=dtype, device=device)
    with torch.no_grad():
        vector_to_parameters(w, net.parameters())

    loss_val, _ = compute_loss(net, X_int, X_ic, bc_data)

    # Optional power transform
    loss_eff = loss_val if power == 1.0 else loss_val ** (1.0 / power)

    params = list(net.parameters())
    grads = torch.autograd.grad(loss_eff, params,
                                create_graph=False, retain_graph=False,
                                allow_unused=False)
    g_flat = torch.cat([g.reshape(-1) for g in grads]).detach().cpu().numpy()
    return float(loss_eff.detach().cpu()), g_flat


# ---- RAD (Residual-Adaptive Distribution) ----------------------------------

def adaptive_rad(net, n_int, rad_args, n_cand=50000):
    """
    Resample interior points weighted by | PDE residual |.

    High-residual regions get more points → faster convergence in
    areas where the network currently performs worst.
    """
    X_cand = sample_interior(n_cand)
    k1, k2 = rad_args
    Y = torch.abs(compute_pde_residual(net, X_cand)[1]).detach().reshape(-1)
    w = Y ** k1
    p = (w / w.mean() + k2)
    p = (p / p.sum()).clamp_min(1e-12)
    ids = torch.multinomial(p, num_samples=n_int, replacement=False)
    return X_cand[ids]

In [ ]:
# ============================================================================
# CELL 7: PHASE 1 — ADAM TRAINING (FIRST-ORDER WARMUP)
# ============================================================================
# Adam is fast per-iteration but only uses gradient info (no curvature).
# We use it to move the network into a roughly good region before switching
# to the more powerful (but expensive) quasi-Newton optimizer.
#
# If Nepochs_ADAM == 0, this phase is skipped entirely (warmup_0 config).
# ============================================================================

# ---- Generate initial training data ---------------------------------------
X_int = sample_interior(Nint)
X_ic  = sample_initial(N0)
bc_data = sample_boundary_periodic(Nb)

adam_losses = []
adam_t0 = perf_counter()

if Nepochs_ADAM > 0:
    # ---- Configure Adam ----------------------------------------------------
    optimizer = torch.optim.Adam(
        net.parameters(), lr=adam_lr, betas=adam_betas, eps=adam_eps
    )
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda step: lr_decay_rate ** (step / lr_decay_steps)
    )

    # ---- Training loop -----------------------------------------------------
    for epoch in range(Nepochs_ADAM):
        # Periodically resample collocation points (RAD if enabled)
        if (epoch + 1) % Nresample == 0:
            if rad_on:
                X_int = adaptive_rad(net, Nint, rad_args, Nsampling)
            else:
                X_int = sample_interior(Nint)
            X_ic    = sample_initial(N0)
            bc_data = sample_boundary_periodic(Nb)

        loss_v, pde_v = adam_step(net, optimizer, X_int, X_ic, bc_data)
        scheduler.step()
        adam_losses.append(loss_v)

        if (epoch + 1) % Nprint == 0:
            print(f"Adam  epoch {epoch+1:5d}/{Nepochs_ADAM}  "
                  f"loss={loss_v:.4e}  pde={pde_v:.4e}")

adam_time = perf_counter() - adam_t0
if adam_losses:
    print(f"\nAdam phase complete: {adam_time:.1f}s  "
          f"final loss={adam_losses[-1]:.4e}")
else:
    print("Adam phase skipped (0 epochs).")

In [ ]:
# ============================================================================
# CELL 8: PHASE 2 — QUASI-NEWTON TRAINING (BFGS / L-BFGS-B)
# ============================================================================
# After Adam warmup, switch to a quasi-Newton optimizer.
#
# Supports two families (controlled by bfgs_method in YAML):
#
#   "BFGS"     — Dense Hessian inverse.  Variants (via bfgs_variant):
#                BFGS_scipy, SSBFGS_OL, SSBFGS_AB, SSBroyden1, SSBroyden2
#                Uses patched SciPy with warm-started H_k between batches.
#
#   "L-BFGS-B" — Limited-memory.  No dense Hessian; stores `maxcor`
#                (s, y) pairs instead.  Much less memory, no warm-start.
#
# If Nbfgs == 0 (Adam-only mode), this cell is effectively a no-op.
#
# Key design: scipy.optimize.minimize is called in a LOOP. Each call runs
# at most Nchange iterations, after which we resample collocation points
# (RAD) and continue.  For dense BFGS, the inverse Hessian H_k is
# preserved across batches (warm-start).
# ============================================================================

if Nbfgs > 0:
    # ---- Setup -------------------------------------------------------------
    initial_weights = parameters_to_vector(
        [p.detach() for p in net.parameters()]
    ).cpu().numpy()

    # Dense Hessian for BFGS variants (ignored by L-BFGS-B)
    use_dense_hessian = (bfgs_method == "BFGS")
    if use_dense_hessian:
        H0 = np.eye(initial_weights.size, dtype=np.float64)

    cont = 0                               # global BFGS iteration counter
    n_ckpts = Nbfgs // Nprint
    bfgs_losses  = np.zeros(n_ckpts)
    bfgs_pde     = np.zeros(n_ckpts)

    initial_scale = bfgs_init_scale
    power = bfgs_power

    # ---- Callback (runs after every BFGS iteration) ------------------------
    def bfgs_callback(*, intermediate_result):
        """Log loss every Nprint iterations."""
        global cont
        cont += 1
        if cont % Nprint != 0:
            return
        idx = cont // Nprint - 1
        if idx < 0 or idx >= n_ckpts:
            return
        loss_val = float(intermediate_result.fun)
        if power != 1.0:
            loss_val = loss_val ** power       # undo power transform for logging
        bfgs_losses[idx] = loss_val
        optimizer_label = bfgs_variant if use_dense_hessian else bfgs_method
        print(f"{optimizer_label}  iter {cont:5d}/{Nbfgs}  loss={loss_val:.4e}")

    # ---- Build method-specific options dict --------------------------------
    def build_bfgs_options():
        """Construct the options dict for scipy.optimize.minimize."""
        opts = {"maxiter": Nchange, "gtol": 0}
        if bfgs_method == "BFGS":
            # Patched SciPy: dense Hessian with self-scaling variants
            opts["hess_inv0"]     = H0
            opts["method_bfgs"]   = bfgs_variant
            opts["initial_scale"] = initial_scale
        elif bfgs_method == "L-BFGS-B":
            # Limited-memory BFGS: no dense Hessian, uses maxcor corrections
            opts["maxcor"]  = bfgs_maxcor
            opts["ftol"]    = 0
        return opts

    # ---- Main BFGS loop ----------------------------------------------------
    bfgs_t0 = perf_counter()

    while cont < Nbfgs:
        result = minimize(
            scipy_loss_and_grad,
            initial_weights,
            args=(net, X_int, X_ic, bc_data, power),
            method=bfgs_method,
            jac=True,
            options=build_bfgs_options(),
            tol=0,
            callback=bfgs_callback,
        )

        # ---- Extract results from this batch -------------------------------
        initial_weights = result.x

        if use_dense_hessian:
            H0 = result.hess_inv
            H0 = 0.5 * (H0 + H0.T)               # symmetrise

            # Positive-definiteness check — reset if degenerate
            try:
                cholesky(H0)
            except LinAlgError:
                print("  [!] H lost positive-definiteness — resetting to I")
                H0 = np.eye(len(initial_weights), dtype=np.float64)

        # ---- Resample collocation points (RAD) -----------------------------
        if rad_on:
            X_int = adaptive_rad(net, Nint, rad_args, Nsampling)
        else:
            X_int = sample_interior(Nint)
        X_ic    = sample_initial(N0)
        bc_data = sample_boundary_periodic(Nb)

        initial_scale = False                  # only on first batch (if at all)

    bfgs_time = perf_counter() - bfgs_t0
    optimizer_label = bfgs_variant if use_dense_hessian else bfgs_method
    print(f"\n{optimizer_label} phase complete: {bfgs_time:.1f}s")
    print(f"Total training time: {adam_time + bfgs_time:.1f}s")

else:
    # ---- Adam-only mode: no second-order phase -----------------------------
    bfgs_time = 0.0
    n_ckpts = 0
    bfgs_losses = np.array([])
    bfgs_pde    = np.array([])
    print(f"\nAdam-only mode: no BFGS phase.  Total time: {adam_time:.1f}s")

In [ ]:
# ============================================================================
# CELL 9: VISUALIZATION AND RESULTS
# ============================================================================
# 1. Loss curve (Adam + BFGS)
# 2. Predicted U(t, x, y) snapshots at selected times
# 3. Initial condition comparison
# ============================================================================

os.makedirs(RESULTS_DIR, exist_ok=True)
optimizer_label = (bfgs_variant if bfgs_method == "BFGS" else bfgs_method) if Nbfgs > 0 else "none"

# ---- 1. Loss curve --------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(range(1, len(adam_losses) + 1), adam_losses, label="Adam", alpha=0.8)
if Nbfgs > 0:
    bfgs_epochs = Nepochs_ADAM + np.arange(1, n_ckpts + 1) * Nprint
    valid = bfgs_losses > 0
    if valid.any():
        ax.semilogy(bfgs_epochs[valid], bfgs_losses[valid],
                     label=optimizer_label, alpha=0.8)
    ax.axvline(Nepochs_ADAM, color="gray", ls="--", lw=0.8, label="Adam→BFGS")
ax.set_xlabel("Iteration")
ax.set_ylabel("Total Loss")
ax.set_title(f"Cahn-Hilliard PINN — {cfg['experiment']['name']}")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "loss_curve.png"), dpi=150)
plt.show()

# ---- 2. Solution snapshots ------------------------------------------------
n_snap = 5
snap_times = np.linspace(t_min, t_max, n_snap)
nx_plot, ny_plot = 64, 64
xs_plot = np.linspace(x_min, x_max, nx_plot)
ys_plot = np.linspace(y_min, y_max, ny_plot)
xx, yy = np.meshgrid(xs_plot, ys_plot)

fig, axes = plt.subplots(1, n_snap, figsize=(4 * n_snap, 3.5))
net.eval()
for i, t_val in enumerate(snap_times):
    tt = np.full_like(xx, t_val)
    X_plot = torch.tensor(
        np.column_stack([tt.ravel(), xx.ravel(), yy.ravel()]),
        dtype=torch.get_default_dtype(), device=DEVICE,
    )
    with torch.no_grad():
        U_plot = forward_pass(net, X_plot).cpu().numpy().reshape(ny_plot, nx_plot)
    im = axes[i].pcolormesh(xs_plot, ys_plot, U_plot, cmap="RdBu_r", shading="auto")
    axes[i].set_title(f"t = {t_val:.1f}")
    axes[i].set_aspect("equal")
    fig.colorbar(im, ax=axes[i], fraction=0.046)
net.train()
fig.suptitle("Predicted U(t, x, y)", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "snapshots.png"), dpi=150, bbox_inches="tight")
plt.show()

# ---- 3. IC comparison -----------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
# True IC (interpolated)
X_ic_plot = torch.tensor(
    np.column_stack([np.zeros(nx_plot * ny_plot), xx.ravel(), yy.ravel()]),
    dtype=torch.get_default_dtype(),
)
U_ic_true_plot = eval_ic(X_ic_plot).numpy().reshape(ny_plot, nx_plot)
im0 = axes[0].pcolormesh(xs_plot, ys_plot, U_ic_true_plot, cmap="RdBu_r", shading="auto")
axes[0].set_title("True IC  U₀(x,y)")
axes[0].set_aspect("equal")
fig.colorbar(im0, ax=axes[0], fraction=0.046)

# Predicted IC
net.eval()
X_ic_dev = X_ic_plot.to(DEVICE)
with torch.no_grad():
    U_ic_pred_plot = forward_pass(net, X_ic_dev).cpu().numpy().reshape(ny_plot, nx_plot)
net.train()
im1 = axes[1].pcolormesh(xs_plot, ys_plot, U_ic_pred_plot, cmap="RdBu_r", shading="auto")
axes[1].set_title("PINN at t = 0")
axes[1].set_aspect("equal")
fig.colorbar(im1, ax=axes[1], fraction=0.046)

fig.suptitle("Initial Condition Comparison", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "ic_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

# ---- 4. Summary -----------------------------------------------------------
print(f"\nExperiment: {cfg['experiment']['name']}")
print(f"  Adam time : {adam_time:.1f}s  ({Nepochs_ADAM} epochs)")
if Nbfgs > 0:
    print(f"  BFGS time : {bfgs_time:.1f}s  ({Nbfgs} iters, {optimizer_label})")
print(f"  Total time: {adam_time + bfgs_time:.1f}s")
print(f"  Final loss: {adam_losses[-1]:.4e}" if Nbfgs == 0 else
      f"  Final BFGS: {bfgs_losses[bfgs_losses > 0][-1]:.4e}" if (bfgs_losses > 0).any() else
      f"  Final loss: {adam_losses[-1]:.4e}")
print(f"Results saved to {RESULTS_DIR}/")